In [8]:
import geopandas as gpd
import numpy as np

In [9]:
mapa = gpd.read_file('../Data/shp/Distrital INEI 2023 geogpsperu SuyoPomalia.shp')

In [10]:
mapa.head(5)

,UBIGEO,CCDD,CCPP,CCDI,DEPARTAMEN,PROVINCIA,DISTRITO,OBJECTID,ESRI_OID,geometry
0,010101,01,01,01,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,1.0,1.0,"POLYGON ((-77.8858 -6.1778, -77.88323 -6.17846..."
1,010102,01,01,02,AMAZONAS,CHACHAPOYAS,ASUNCION,2.0,2.0,"POLYGON ((-77.74482 -5.94497, -77.74482 -5.945..."
2,010103,01,01,03,AMAZONAS,CHACHAPOYAS,BALSAS,3.0,3.0,"POLYGON ((-77.9358 -6.69039, -77.93531 -6.6909..."
3,010104,01,01,04,AMAZONAS,CHACHAPOYAS,CHETO,4.0,4.0,"POLYGON ((-77.71486 -6.24598, -77.71485 -6.245..."
4,010105,01,01,05,AMAZONAS,CHACHAPOYAS,CHILIQUIN,5.0,5.0,"POLYGON ((-77.77405 -5.99598, -77.77328 -5.996..."


In [12]:
mapa.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [24]:
mapa.centroid.x, mapa.centroid.y

C:\Users\encue\AppData\Local\Temp\ipykernel_23760\990197039.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  mapa.centroid.x, mapa.centroid.y


(0      -77.856488
 1      -77.743204
 2      -77.949166
 3      -77.677166
 4      -77.755866
           ...    
 1886   -75.049032
 1887   -75.047983
 1888   -75.414415
 1889   -75.752348
 1890   -71.582316
 Length: 1891, dtype: float64,
 0       -6.247743
 1       -5.996133
 2       -6.807726
 3       -6.297970
 4       -6.078208
           ...    
 1886    -8.641301
 1887    -8.824933
 1888    -8.797873
 1889    -9.035813
 1890   -10.373495
 Length: 1891, dtype: float64)

In [14]:
# Reproject the GeoDataFrame to a CRS that uses meters (e.g., EPSG:3395)
mapa = mapa.to_crs(epsg=3857)

# Calculate the area in square meters
mapa['area_m2'] = mapa.geometry.area

# Convert the area from square meters to square kilometers
mapa['area_km2'] = mapa['area_m2'] / 1e6

# Check the total area
total_area_km2 = mapa['area_km2'].sum()

print(f"Total Area: {total_area_km2} km²")

Total Area: 1336344.953805421 km²


In [15]:
# Reproyectar el GeoDataFrame a un CRS UTM (por ejemplo, EPSG:32718 para Perú occidental)
mapa = mapa.to_crs(epsg=32718)

# Calcular el área en metros cuadrados
mapa['area_m2'] = mapa.geometry.area

# Convertir el área de metros cuadrados a kilómetros cuadrados
mapa['area_km2'] = mapa['area_m2'] / 1e6

# Ver el área total
total_area_km2 = mapa['area_km2'].sum()

print(f"Área total: {total_area_km2} km²")

Área total: 1288518.8351335437 km²


In [23]:
mapa = gpd.read_file('../Data/shp/Distrital INEI 2023 geogpsperu SuyoPomalia.shp')

def get_utm_epsg(lon, lat):
    # 1. Calcular la zona (1 a 60)
    zone = int((lon + 180) // 6) + 1
    
    # 2. Determinar el prefijo según el hemisferio usando la LATITUD
    # EPSG 326XX para el Norte (lat >= 0)
    # EPSG 327XX para el Sur (lat < 0)
    if lat >= 0:
        return 32600 + zone
    else:
        return 32700 + zone

# Actualizamos la creación de la columna utm_zone usando latitud y longitud
# Usamos el centroide de la geometría para obtener ambos valores
mapa['utm_zone'] = mapa.geometry.apply(
    lambda x: get_utm_epsg(x.centroid.x, x.centroid.y)
)

def reproject_geometry(row):
    # Verificamos que el mapa original tenga un CRS definido
    if mapa.crs is None:
        raise ValueError("El GeoDataFrame 'mapa' no tiene un CRS definido. Define uno antes de proyectar (ej. mapa.set_crs(epsg=4326)).")
        
    utm_zone = int(row['utm_zone'])
    # Creamos la GeoSeries temporal y proyectamos
    temp_series = gpd.GeoSeries([row.geometry], crs=mapa.crs)
    return temp_series.to_crs(epsg=utm_zone).iloc[0]

# Aplicar la reproyección
mapa['geometry_utm'] = mapa.apply(reproject_geometry, axis=1)

# El resto del cálculo de área funcionará correctamente
mapa['area_m2'] = mapa['geometry_utm'].area
mapa['area_km2'] = mapa['area_m2'] / 1e6

# Ver el área total
total_area_km2 = mapa['area_km2'].sum()

print(f"Área total: {total_area_km2} km²")

Área total: 1286603.657379727 km²
